In [29]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException   
from datetime import datetime
import traceback
from random import randint
import pandas as pd
import re
import numpy as np
from time import sleep
import urllib.parse

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

In [30]:
# Set up functions
def print_update(func):
    def wrapper(*args, **kwargs):
        print(f'Running {func.__name__}')
        return func(*args, **kwargs)
    return wrapper


def reject_cookies(driver, temp=0, retries=2):
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.ID, 'onetrust-reject-all-handler')))
        driver.find_element(By.ID, 'onetrust-reject-all-handler').click()
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            print(f'{retries} attempts failed to Reject cookies')
        else:
            reject_cookies(driver, temp + 1)


def skip_tutorial(driver, temp=0, retries=2):
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip')))
        driver.find_element(By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip').click()
        driver.find_element(By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip').click()
        driver.find_element(By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip').click()
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            print(f'{temp} attempts failed to Skip the tutorial')
        else:
            skip_tutorial(driver, temp + 1)


def setup_driver():
    options = webdriver.ChromeOptions()
    prefs = {"profile.managed_default_content_settings.images": 2}
    options.add_experimental_option("prefs", prefs)
    options.add_argument("--headless")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--log-level=4")
    options.add_argument("--disable-extensions")
    driver = webdriver.Chrome(options=options)
    driver.maximize_window()

    driver.get('https://es.wallapop.com/app/search?filters_source=category_navigation&latitude=40.41956&longitude=-3.69196')

    reject_cookies(driver)
    skip_tutorial(driver)

    return driver

@print_update
def get_explorer_urls(product_id=None, category_id=24200):
    # gets the corresponding urls to wallapop listing explorer
    base = 'https://es.wallapop.com/app/search?'
    urls = []

    if product_id:
        pass ## Do later
        # url = url if (product_id == None) else url + f'keywords={urllib.parse.quote(product_id)}&'
    else:
        '''
        Make a df for every category 
        Get the histogram curve for each category type by querying the listings db
        '''


        if category_id == 24200:
            object_type_ids = [9447, 9487, 10175, 24102]
            object_type_ids = [9447]
            for object_id in object_type_ids:
                url = base + f'category_ids={category_id}&filters_source=quick_filters&latitude=40.41956&longitude=-3.69196&object_type_ids={object_id}&'
                urls += split_explorer_url(url, product_id=product_id, category_id=category_id, object_id=object_id)
        else:
            print('category not supported')

    return urls


def split_explorer_url(url, product_id=None, category_id=None, object_id=None):
    # breaks up url into smaller subsets to divide up scrape load
    urls = []

    if product_id:
        # IGNORE write later after figuring out product_id classification module
        pass
    else:
        start, increment, loops = 0, 4, 12
        current_range = increment

        for i in range(loops):
            min = start
            max = start + current_range

            if i == 0:
                pricequery = f'max_sale_price={max}'
            elif i + 1 == len(range(loops)):
                pricequery = f'min_sale_price={min+.01}'
            else:
                pricequery = f'min_sale_price={min+.01}&max_sale_price={max}'

            start += current_range
            current_range += increment + (i-1)**2 # Ramps up range ~according to previous scrapes

            urls.append({
                'explorer_url': url + pricequery,
                'product_id': product_id,
                'category_id': category_id,
                'object_id': object_id,
                'date_scraped': np.nan,
                'scrape_empty': np.nan,
                'scrape_incomplete': np.nan,
                'redundant_url': np.nan,
            })
    return urls


def load_more(driver, temp=0, retries=2):
    # Clicks button that initiates infinite scroll dynamic listings
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.ID, 'btn-load-more')))
        driver.find_element(By.ID, 'btn-load-more').click()
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            print(f'{temp} attempts failed to Click the load button')
        else:
            load_more(driver, temp + 1)


def wait_listings(driver, temp=0, retries=3):
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'a.ItemCardList__item')))
        return True
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            print(f'{temp} attempts failed to Find listings')
        else:
            wait_listings(driver, temp + 1)

In [31]:
# Explorer function
def scrape_explorer(driver, object_id=None):
    # Scrapes listings displayed in explorer
    scrapings = []
    cards = driver.find_elements(By.CSS_SELECTOR, 'a.ItemCardList__item')
    for card in cards:
        title = card.get_attribute('title')
        href = card.get_attribute('href')
        href = href.split('https://es.wallapop.com/item/')[-1]
        listing_id = int(href.split('-')[-1])
        num_images = len(card.find_elements(By.TAG_NAME, 'li'))
        if not num_images:
            num_images = 1
        date_scraped = int(datetime.now().strftime('%y%m%d%H%M%S'))

        price = card.find_element(By.CSS_SELECTOR, 'span.ItemCard__price').text.strip()
        price = float(re.sub(r'[^\d,]', '', price).replace(',', '.'))
        price = int(price*100)
        
        try:
            if card.find_element(By.CSS_SELECTOR, 'p.ItemCard__highlight-text.pb-2'):
                featured = 1
            else:
                featured = 0
        except NoSuchElementException:
            featured = 0

        shadow_hosts = card.find_elements(By.CSS_SELECTOR, 'wallapop-badge')
        shipping, reserved, walla_pro = 0, 0, 0
        for shadow_host in shadow_hosts:
            shadow_root = driver.execute_script('return arguments[0].shadowRoot', shadow_host)
            span = shadow_root.find_element(By.CSS_SELECTOR, 'span').text.strip() 
            if span == 'Sólo venta en persona':
                shipping = 0
            elif span == 'Envío disponible':
                shipping = 1
            elif span == 'Envío gratis':
                shipping = 2
            elif span == 'Reservado':
                reserved = 1
            elif span == 'Reacondicionado':
                walla_pro = 1
            else:
                print('\nUNEXPECTED SHADOW_ROOT DATA\n')
                print(card)

        scrapings.append({
            'title': title,
            'https://es.wallapop.com/item/': href,
            'object_id': object_id,
            'listing_id': listing_id,
            'num_images': num_images,
            'date_last_scraped': date_scraped,
            'date_first_scraped': date_scraped,
            'price_cents': price,
            'featured': featured,
            'shipping': shipping,
            'reserved': reserved,
            'walla_pro': walla_pro,
        })
        
    return pd.DataFrame(scrapings)


def scroll_explorer(driver, category_id=None, object_id=None):
    global scraped_df
    global max_scrolls
    previous_height = driver.execute_script("return document.body.scrollHeight")
    sleep_time = 0.1
    count = 0
    timer = datetime.now()
    complete = False

    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        count += 1
        if count == max_scrolls:
            # print('Scroll: Count reached')
            card = driver.find_elements(By.CSS_SELECTOR, 'a.ItemCardList__item')[-1]
            # WebDriverWait(driver, 5).until(EC.element_to_be_clickable(card))
            sleep(sleep_time * 3)
            scraped_df, complete = check_scraping(driver, False, timer, category_id=category_id, object_id=object_id)

        try:
            WebDriverWait(driver, sleep_time).until(lambda driver: driver.execute_script("return document.body.scrollHeight") != previous_height)
            sleep_time /= 2
            previous_height = driver.execute_script("return document.body.scrollHeight")
        except TimeoutException:
            sleep_time += sleep_time
            if sleep_time > 12:
                print('Scroll: Timed out')
                card = driver.find_elements(By.CSS_SELECTOR, 'a.ItemCardList__item')[-1]
                WebDriverWait(driver, 5).until(EC.element_to_be_clickable(card))
                scraped_df, complete = check_scraping(driver, True, timer, category_id=category_id, object_id=object_id)

        if complete:
            return scraped_df, True


def check_scraping(driver, timed_out, timer=None, category_id=None, object_id=None):
    global scraped_df       

    # Get scrapings and check if they already exist in global
    scrapings_df = scrape_explorer(driver, object_id=object_id)

    if scrapings_df.empty:
        print('EMPTY SCRAPINGS')
        return scraped_df, True
    else:
        scraped_df, redundant = update_scraping(scrapings_df, category_id=category_id, object_id=object_id)
        if redundant:
            return scraped_df, True
        clear_explorer(driver)

        # # Print stats
        # timespent = datetime.now() - timer
        # efficiency = round(len(scrapings_df)/timespent.total_seconds(), 3)
        # print(f'    scrapings took: {timespent}')
        # print(f'    scrapings count: {len(scrapings_df)}')
        # print(f'    scrapings efficiency = {efficiency}/sec')

        if timed_out:
            return scraped_df, True
        else:
            return scroll_explorer(driver, category_id=category_id, object_id=object_id)


def update_scraping(scrapings_df, category_id=None, object_id=None):
    global scraped_df

    try:
        object_df = pd.read_csv(f'data/categories/{category_id}/{object_id}.csv')
        original_len = len(object_df)
        ignore_columns = ['date_first_scraped', 'date_last_scraped']
        compare_columns = [col for col in scrapings_df.columns if col not in ignore_columns]

        for _, row in scrapings_df.iterrows():
            match = object_df[compare_columns].eq(row[compare_columns]).all(axis=1)
            if match.any():
                object_df.loc[match, 'date_last_scraped'] = int(datetime.now().strftime('%y%m%d%H%M%S'))
            else:
                object_df = pd.concat([object_df, pd.DataFrame([row])], ignore_index=True).drop_duplicates()
        
        object_df.to_csv(f'data/categories/{category_id}/{object_id}.csv', index=False)

        if len(object_df) == original_len:
            print('REDUNDANT SCRAPINGS')
            return scraped_df, True
        elif 'scraped_df' not in globals():
            scraped_df = scrapings_df
        else:
            scraped_df = pd.concat([scraped_df, scrapings_df]).drop_duplicates()
            
        return scraped_df, False

    except FileNotFoundError:
        object_df = scrapings_df
        object_df.to_csv(f'data/categories/{category_id}/{object_id}.csv', index=False)
        return scrapings_df, False


def clear_explorer(driver):
    driver.execute_script("window.scrollTo(0, 0);")
    try:
        # Find elements and delete all siblings
        element = driver.find_element(By.CSS_SELECTOR, 'a.ItemCardList__item')
        parent = element.find_element(By.XPATH, '..')
        siblings = parent.find_elements(By.XPATH, '*')
        for sibling in siblings:
            driver.execute_script("arguments[0].remove();", sibling)
    except NoSuchElementException:
        print('    No listings found to clear to clear')
   

def check_newest(category_ids=[24200, 12579, 13100]):
    global max_scrolls
    global scraped_df
    all_scraped = 0
    best_efficiency = 0
    best_scrolls = 0

    for category_id in category_ids:
        scraped_count = 0
        category_start = datetime.now()

        try:
            categories = pd.read_csv('data/categories/categories.csv')
            object_type_ids = categories[categories['category_id'] == category_id]
            object_type_ids = object_type_ids[object_type_ids['subgroup_name'] != 'FULL_GROUP']
            # object_type_ids = object_type_ids[object_type_ids['last_checked'] != int(category_start.strftime('%y%m%d'))]
        except FileNotFoundError:
            return 'data/categories/categories.csv is missing'
        category_name = categories.loc[categories['category_id'] == category_id, 'category_name'].values[0]
        print(f'\n=== Checking {category_name}')

        for object_id in object_type_ids['object_type_id']:
            object_name = categories.loc[categories['object_type_id'] == object_id, 'subgroup_name'].values[0]
            print(f'\nSCRAPING {object_name.upper()} {object_id}')

            object_start = datetime.now()
            driver = setup_driver()
            driver.get(f'https://es.wallapop.com/app/search?object_type_ids={object_id}&category_ids={category_id}&filters_source=quick_filters&latitude=40.41956&longitude=-3.69196&order_by=newest')
            if wait_listings(driver):
                try:
                    load_more(driver)
                    scraped_df = pd.DataFrame()
                    scraped_df, _ = scroll_explorer(driver, category_id=category_id, object_id=object_id)
                    driver.quit()
                except Exception as e:
                    print(f'{object_name} {object_id} Error: {e}')
                    traceback.print_exc()
                    driver.quit()
            else:
                driver.quit()

            # Update categories.csv
            categories.loc[categories['object_type_id'] == object_id, 'last_checked'] = int(datetime.now().strftime('%y%m%d'))
            categories.to_csv('data/categories/categories.csv', index=False)

            timespent = datetime.now() - object_start
            scraped_count += len(scraped_df)
            efficiency = round(len(scraped_df)/timespent.total_seconds(), 3)

            # # Scroll optimizer
            # if efficiency > best_efficiency:
            #     best_efficiency = efficiency
            #     best_scrolls = max_scrolls
            #     max_scrolls += 1
            # else:
            #     max_scrolls -= 1

            # max_scrolls = max(4, max_scrolls)

            print(f'\n{object_name.upper()} {object_id} SCRAPE COMPLETE')
            print(f'    {object_name} took: {timespent}')
            print(f'    {object_name} scraped: {len(scraped_df)}')
            print(f'    {object_name} efficency: {efficiency}/sec')
            print(f'    Max scrolls: {max_scrolls}, Optimal: {best_scrolls}')

        # Update fullgroup in category.csv
        categories.loc[(categories['category_id'] == category_id) & (categories['subgroup_name'] == 'FULL_GROUP'), 'last_checked'] = int(datetime.now().strftime('%y%m%d'))
        categories.to_csv('data/categories/categories.csv', index=False)

        timespent = datetime.now() - category_start
        all_scraped += scraped_count
        print(f'\n{category_name} CATEGORY checked')
        print(f'    {category_name} took: {timespent}')
        print(f'    {category_name} scraped: {scraped_count}')
        print(f'    {category_name} efficency: {round(scraped_count/timespent.total_seconds(), 3)}/sec')

    if all_scraped == 0:
        return '\n=== Already checked for today ===\n'
    print('\n=== All categories checked ===\n')

In [32]:
categories = [24200]
max_scrolls = 10
scraped_df = pd.DataFrame()

while True:
    try:
        message = check_newest(categories)
        if message:
            print(message)
            break
    except Exception:
        print('error outside')
        # traceback.print_exc()


=== Checking Tecnología y electrónica

SCRAPING TV 10414
TV 10414 Error: Message: unknown error: session deleted because of page crash
from unknown error: cannot determine loading status
from tab crashed
  (Session info: chrome-headless-shell=127.0.6533.100)
Stacktrace:
0   chromedriver                        0x00000001046190b8 cxxbridge1$str$ptr + 1887276
1   chromedriver                        0x0000000104611794 cxxbridge1$str$ptr + 1856264
2   chromedriver                        0x0000000104220694 cxxbridge1$string$len + 88116
3   chromedriver                        0x000000010420cf58 cxxbridge1$string$len + 8440
4   chromedriver                        0x000000010420c4b0 cxxbridge1$string$len + 5712
5   chromedriver                        0x000000010420bc30 cxxbridge1$string$len + 3536
6   chromedriver                        0x000000010420bb7c cxxbridge1$string$len + 3356
7   chromedriver                        0x0000000104209954 core::str::slice_error_fail::he7b2aa4898bc357e + 599

Traceback (most recent call last):
  File "/var/folders/35/xfdc6bm10c98tsfyqbkl8vq80000gn/T/ipykernel_46044/534676598.py", line 207, in check_newest
    scraped_df, _ = scroll_explorer(driver, category_id=category_id, object_id=object_id)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/35/xfdc6bm10c98tsfyqbkl8vq80000gn/T/ipykernel_46044/534676598.py", line 82, in scroll_explorer
    scraped_df, complete = check_scraping(driver, False, timer, category_id=category_id, object_id=object_id)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/35/xfdc6bm10c98tsfyqbkl8vq80000gn/T/ipykernel_46044/534676598.py", line 125, in check_scraping
    return scroll_explorer(driver, category_id=category_id, object_id=object_id)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/35/xfdc6bm10c98tsfyqbkl8vq80000gn/T/


TV 10414 SCRAPE COMPLETE
    TV took: 0:31:02.978100
    TV scraped: 24801
    TV efficency: 13.313/sec
    Max scrolls: 10, Optimal: 0

SCRAPING PROYECTORES 10406
Scroll: Timed out
Proyectores 10406 Error: list index out of range


Traceback (most recent call last):
  File "/var/folders/35/xfdc6bm10c98tsfyqbkl8vq80000gn/T/ipykernel_46044/534676598.py", line 85, in scroll_explorer
    WebDriverWait(driver, sleep_time).until(lambda driver: driver.execute_script("return document.body.scrollHeight") != previous_height)
  File "/Users/alejandroyepesshutova/Documents/testing/venv/lib/python3.12/site-packages/selenium/webdriver/support/wait.py", line 105, in until
    raise TimeoutException(message, screen, stacktrace)
selenium.common.exceptions.TimeoutException: Message: 


During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/var/folders/35/xfdc6bm10c98tsfyqbkl8vq80000gn/T/ipykernel_46044/534676598.py", line 207, in check_newest
    scraped_df, _ = scroll_explorer(driver, category_id=category_id, object_id=object_id)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/35/xfdc6bm10c98tsfyqbkl8vq80000gn/


PROYECTORES 10406 SCRAPE COMPLETE
    Proyectores took: 0:15:57.582566
    Proyectores scraped: 15118
    Proyectores efficency: 15.788/sec
    Max scrolls: 10, Optimal: 0

SCRAPING ACCESORIOS DE TELEVISORES Y PROYECTORES 10405
Accesorios de televisores y proyectores 10405 Error: Message: unknown error: session deleted because of page crash
from unknown error: cannot determine loading status
from tab crashed
  (Session info: chrome-headless-shell=127.0.6533.100)
Stacktrace:
0   chromedriver                        0x0000000104fcd0b8 cxxbridge1$str$ptr + 1887276
1   chromedriver                        0x0000000104fc5794 cxxbridge1$str$ptr + 1856264
2   chromedriver                        0x0000000104bd4694 cxxbridge1$string$len + 88116
3   chromedriver                        0x0000000104bbfb08 cxxbridge1$string$len + 3240
4   chromedriver                        0x0000000104bbd954 core::str::slice_error_fail::he7b2aa4898bc357e + 59976
5   chromedriver                        0x0000000104b

Traceback (most recent call last):
  File "/var/folders/35/xfdc6bm10c98tsfyqbkl8vq80000gn/T/ipykernel_46044/534676598.py", line 207, in check_newest
    scraped_df, _ = scroll_explorer(driver, category_id=category_id, object_id=object_id)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/35/xfdc6bm10c98tsfyqbkl8vq80000gn/T/ipykernel_46044/534676598.py", line 82, in scroll_explorer
    scraped_df, complete = check_scraping(driver, False, timer, category_id=category_id, object_id=object_id)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/35/xfdc6bm10c98tsfyqbkl8vq80000gn/T/ipykernel_46044/534676598.py", line 125, in check_scraping
    return scroll_explorer(driver, category_id=category_id, object_id=object_id)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/35/xfdc6bm10c98tsfyqbkl8vq80000gn/T/


ACCESORIOS DE TELEVISORES Y PROYECTORES 10405 SCRAPE COMPLETE
    Accesorios de televisores y proyectores took: 0:33:07.259240
    Accesorios de televisores y proyectores scraped: 25181
    Accesorios de televisores y proyectores efficency: 12.671/sec
    Max scrolls: 10, Optimal: 0

SCRAPING OTROS ARTÍCULOS DE IMAGEN 24192
Scroll: Timed out

OTROS ARTÍCULOS DE IMAGEN 24192 SCRAPE COMPLETE
    Otros artículos de imagen took: 0:29:10.216506
    Otros artículos de imagen scraped: 24440
    Otros artículos de imagen efficency: 13.964/sec
    Max scrolls: 10, Optimal: 0

SCRAPING AURICULARES 10395
Auriculares 10395 Error: Message: unknown error: session deleted because of page crash
from unknown error: cannot determine loading status
from tab crashed
  (Session info: chrome-headless-shell=127.0.6533.100)
Stacktrace:
0   chromedriver                        0x0000000104e610b8 cxxbridge1$str$ptr + 1887276
1   chromedriver                        0x0000000104e59794 cxxbridge1$str$ptr + 1856264

Traceback (most recent call last):
  File "/var/folders/35/xfdc6bm10c98tsfyqbkl8vq80000gn/T/ipykernel_46044/534676598.py", line 207, in check_newest
    scraped_df, _ = scroll_explorer(driver, category_id=category_id, object_id=object_id)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/35/xfdc6bm10c98tsfyqbkl8vq80000gn/T/ipykernel_46044/534676598.py", line 82, in scroll_explorer
    scraped_df, complete = check_scraping(driver, False, timer, category_id=category_id, object_id=object_id)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/35/xfdc6bm10c98tsfyqbkl8vq80000gn/T/ipykernel_46044/534676598.py", line 125, in check_scraping
    return scroll_explorer(driver, category_id=category_id, object_id=object_id)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/35/xfdc6bm10c98tsfyqbkl8vq80000gn/T/


AURICULARES 10395 SCRAPE COMPLETE
    Auriculares took: 0:29:01.602985
    Auriculares scraped: 24256
    Auriculares efficency: 13.927/sec
    Max scrolls: 10, Optimal: 0

SCRAPING ALTAVOCES 24150


KeyboardInterrupt: 

In [ ]:
m5 = [24.428, 23.828]

m10 = [23.48, ]

m20 = [22.24, ]

m30 = [22.291,]

m50 = [20.362, 19.662, 19.291, 18.17, 17.483, 17.87, 17.45, 17.552, 17.203, 17.308, 15.899, 14.984, 15.242, 14.274, 14.076, 13.284, 12.064, 12.274, 10.977, 11.196, 10.177, 10.571, 8.721, 8.599, 7.391]

m100 = [18.235]

